## Sum, Average, Min, Max, and Frequency

In [1]:
import collections

numbers = [1,1,2,2,3,3,4,4,5,5]

def analyze(numbers):
    total = sum(numbers)
    avg = total/len(numbers)
    lowest = min(numbers)
    highest = max(numbers)
    return total, avg, lowest, highest

total, avg, lowest, highest = analyze(numbers)
print(total, avg, lowest, highest)

def frequency(numbers):
    counts = {}
    for number in numbers:
        if number not in counts:
            counts[number] = 1
        else: counts[number] += 1
    return counts
print (frequency(numbers))

30 3.0 1 5
{1: 2, 2: 2, 3: 2, 4: 2, 5: 2}


`analyze` composes Python's built-ins (`sum`, `len`, `min`, `max`) instead of hand-rolling a loop the way you would in Java, and returns all four as a tuple in one shot - unpacked on the calling side into four separate variables. `frequency` counts occurrences using a dict (Python's `HashMap`): not seen yet -> start at 1, already seen -> increment. That `else` branch is easy to forget - skip it and every count silently gets stuck at 1.

## Frequency Counting on a Sorted List

In [2]:
def frequency_sorted(numbers):
    counts = {}
    current = numbers[0]
    count = 1

    for number in numbers[1:]:
        if number == current:
            count += 1
        else:
            counts[current] = count
            current = number
            count = 1

    counts[current] = count
    return counts
print(frequency_sorted(numbers))

{1: 2, 2: 2, 3: 2, 4: 2, 5: 2}


Same goal as `frequency`, exploiting that the list is sorted: duplicates sit next to each other, so this compares each number to a running `current` value instead of doing a dict lookup every time. `counts[current] = count` after the loop is easy to miss - it's the only line that saves the very last group, since the loop only writes a count when it detects a *change*, and the last group never triggers one.

## Binary Search

In [3]:

def binary_search(A, n, x):
    if n == 0:
        return False
    mid = n // 2
    if A[mid] == x:
        return True
    elif A[mid] > x:
        return binary_search(A[0:mid], mid, x)
    else:
        return binary_search(A[mid+1:n], n - mid - 1, x)

numbers = [1, 3, 5, 7, 9, 11]
result = binary_search(numbers, len(numbers), 7)
print(result)   # should print True, since 7 is in the list

result2 = binary_search(numbers, len(numbers), 4)
print(result2)  # should print False, since 4 isn't in the list

True
False


Binary search only works on a **sorted** list - check the middle, throw away the half that can't contain the target, repeat (same idea as always guessing the midpoint in a "guess my number" game). `n == 0` is the base case; without it this crashes or recurses forever once the search space empties out. Passing `mid` / `n - mid - 1` as the next call's size (not `n/2`) matters, since the two halves aren't always equal length.

## Binary Search, Traced

In [4]:
def binary_search(A, n, x):
    print(f"searching {A} for {x}")   # shows the list shrinking each call

    if n == 0:
        return False

    mid = n // 2

    if A[mid] == x:
        return True
    elif A[mid] > x:
        return binary_search(A[0:mid], mid, x)
    else:
        return binary_search(A[mid+1:n], n - mid - 1, x)


numbers = [1, 3, 5, 7, 9, 11]

print("--- looking for 9 (exists) ---")
result = binary_search(numbers, len(numbers), 9)
print("found:", result)

print()
print("--- looking for 4 (doesn't exist) ---")
result2 = binary_search(numbers, len(numbers), 4)
print("found:", result2)

--- looking for 9 (exists) ---
searching [1, 3, 5, 7, 9, 11] for 9
searching [9, 11] for 9
searching [9] for 9
found: True

--- looking for 4 (doesn't exist) ---
searching [1, 3, 5, 7, 9, 11] for 4
searching [1, 3, 5] for 4
searching [5] for 4
searching [] for 4
found: False


Same function with a `print` added so the list visibly shrinks on each call. Searching for 9 narrows a 6-element list down to one element in 3 steps; searching for 4 (absent) narrows all the way to `[]` - exactly the `n == 0` base case catching it and returning `False` instead of crashing.

## Bubble Sort (sortArray)

In [5]:
def bubble_sort(A):
    """Repeatedly swap adjacent out-of-order pairs until the whole array is sorted."""
    n = len(A)
    for i in range(n):
        for j in range(n - i - 1):
            if A[j] > A[j + 1]:
                A[j], A[j + 1] = A[j + 1], A[j]
    return A


numbers = [5, 2, 9, 1, 5, 6]
print("before:", numbers)
print("after: ", bubble_sort(numbers))

before: [5, 2, 9, 1, 5, 6]
after:  [1, 2, 5, 5, 6, 9]


The lecture's pseudocode for `SortArray` actually reuses the *same* variable `i` for
both loops (`for i = 0 to n-1 / for i = 0 to n-1`) - that's a bug, since the inner loop would
overwrite the outer loop's counter every single pass. In Python that's fixed by giving the inner
loop its own name (`j`).

The inner bound `n - i - 1` (instead of just `n-1`, like the pseudocode has) fixes a second issue:
`A[j+1]` needs `j+1` to stay a valid index, so `j` can only go up to `n-2` on the very first pass.
And after each pass, the largest remaining value has always "bubbled" all the way to the end - so
each pass can safely check one element less than the last (hence shrinking by `i` every time).

In [6]:
def bubble_sort_traced(A):
    n = len(A)
    for i in range(n):
        swapped = False
        for j in range(n - i - 1):
            if A[j] > A[j + 1]:
                A[j], A[j + 1] = A[j + 1], A[j]
                swapped = True
        print(f"after pass {i + 1}: {A}")
        if not swapped:
            print("  no swaps this pass -> already sorted, stopping early")
            break
    return A


bubble_sort_traced([5, 2, 9, 1, 5, 6])

after pass 1: [2, 5, 1, 5, 6, 9]
after pass 2: [2, 1, 5, 5, 6, 9]
after pass 3: [1, 2, 5, 5, 6, 9]
after pass 4: [1, 2, 5, 5, 6, 9]
  no swaps this pass -> already sorted, stopping early


[1, 2, 5, 5, 6, 9]

Watch the biggest untouched value each pass: `9` reaches the end after pass 1, `6` after
pass 2, and so on - exactly the "bubbling to the top" the name describes. The early-exit `if not
swapped: break` is a nice practical optimization: bubble sort is `O(n²)` in the worst case (a
reversed array, like `SortArray`'s own analysis on the time-complexity slides), but on an
**already-sorted** array it does a single `O(n)` pass, notices nothing moved, and stops.

## Recursion exercise 1: Factorial

`n!` (n factorial) = `n * (n-1) * (n-2) * ... * 1`, with `0! = 1` by definition. This is a cleaner recursion example than binary search, because the base case and the recursive step are both very explicit.

In [7]:
def factorial(n):
    if n == 0 or n == 1:
        return 1
    return n * factorial(n - 1)


for i in range(6):
    print(f"{i}! = {factorial(i)}")

0! = 1
1! = 1
2! = 2
3! = 6
4! = 24
5! = 120


### Tracing the Recursion

Same idea as the `binary_search` print-trace from earlier - adding indentation that grows with recursion depth makes the call stack visible instead of invisible:

In [8]:
def factorial_traced(n, depth=0):
    indent = "  " * depth
    print(f"{indent}factorial({n}) called")
    if n == 0 or n == 1:
        print(f"{indent}base case reached, returning 1")
        return 1
    result = n * factorial_traced(n - 1, depth + 1)
    print(f"{indent}factorial({n}) returns {result}")
    return result


factorial_traced(4)

factorial(4) called
  factorial(3) called
    factorial(2) called
      factorial(1) called
      base case reached, returning 1
    factorial(2) returns 2
  factorial(3) returns 6
factorial(4) returns 24


24

Notice the two-phase shape: the indentation grows all the way down to the base case (that's every recursive call happening one after another, each one waiting on the next), then the "returns" messages print in REVERSE order coming back up - `factorial(1)` returns first, then `factorial(2)`, then `factorial(3)`, then `factorial(4)` last. That unwinding-in-reverse is true of every recursive function, not just this one, since each call is stuck waiting for the one below it to finish before it can compute its own `return`.

## Recursion exercise 2: Fibonacci

Each number is the sum of the two before it: `0, 1, 1, 2, 3, 5, 8, 13, ...`. Unlike factorial (which only ever calls itself once per call) and binary search (which only ever recurses into ONE half), Fibonacci recurses TWICE per call - this is what a branching recursion tree looks like, rather than a straight line.

In [9]:
def fibonacci(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fibonacci(n - 1) + fibonacci(n - 2)


print([fibonacci(i) for i in range(10)])

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


Worth knowing even without writing the fix yet: this version is very slow for larger `n`, because it ends up recalculating the same smaller Fibonacci numbers over and over (`fibonacci(5)` calls `fibonacci(3)` twice, `fibonacci(2)` three times, and so on). That's a setup for *memoization* (caching results you've already computed) - a natural next step once dictionaries and recursion have both been covered, which they now have.